In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import random
import os
from sklearn.metrics import balanced_accuracy_score

1- image
2- patches
3- patch embedding
4- cls token
5- positional embedding
6- 12x transformer layers
7- CLS output
8- classification head
8- 17 class logits

Theoretical: "Each patch is flattened and projected into a
768-dimensional embedding via a learned linear transformation"
Implementation: a Conv2d with kernel=patch_size, stride=patch_size
is mathematically identical to flattening each patch and
multiplying by a weight matrix — much faster in practice.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.image_size  = image_size
        self.patch_size  = patch_size
        self.num_patches = (image_size // patch_size) ** 2   # 196 patches for 224x224

        # Single Conv2d replaces: flatten(patch) → Linear(768→768)
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        # x: (B, 3, 224, 224)
        x = self.projection(x)       # (B, 768, 14, 14)
        x = x.flatten(2)             # (B, 768, 196)
        x = x.transpose(1, 2)        # (B, 196, 768) — 196 patch embeddings
        return x

Theoretical: "For each of the 12 heads, three separate linear
projections produce Q, K, V. Dot products compute attention
scores, scaled by √64, softmaxed into weights, then used to
compute a weighted sum of Values."

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=768, num_heads=12, dropout=0.0):
        super().__init__()
        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads   # 768 // 12 = 64
        self.scale      = self.head_dim ** -0.5    # 1/√64

        # Single projection for Q, K, V together — split after
        self.qkv     = nn.Linear(embed_dim, embed_dim * 3, bias=True)
        self.proj    = nn.Linear(embed_dim, embed_dim)       # final projection to mix heads
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape   # B=batch, N=197 tokens, C=768

        # Project to Q, K, V and split into heads
        qkv = self.qkv(x)                                          # (B, 197, 768*3)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim)  # (B, 197, 3, 12, 64)
        qkv = qkv.permute(2, 0, 3, 1, 4)                          # (3, B, 12, 197, 64)
        q, k, v = qkv.unbind(0)                                    # each: (B, 12, 197, 64)

        # Attention scores: Q × K^T / √64
        attn = (q @ k.transpose(-2, -1)) * self.scale   # (B, 12, 197, 197)

        # Softmax to get attention weights — each row sums to 1
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        # Weighted sum of Values
        x = (attn @ v)                          # (B, 12, 197, 64)
        x = x.transpose(1, 2).reshape(B, N, C)  # (B, 197, 768) — concatenate heads
        x = self.proj(x)                         # (B, 197, 768) — mix across heads
        return x


Theoretical: "The MLP expands 768 → 3072 via GELU activation,
then projects back to 768. Applied independently to each token."

In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, embed_dim=768, mlp_ratio=4, dropout=0.0):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)   # 768 * 4 = 3072
        self.fc1     = nn.Linear(embed_dim, hidden_dim)
        self.act     = nn.GELU()                  # SigLIP uses GELU, not ReLU
        self.fc2     = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)       # 768 → 3072
        x = self.act(x)       # GELU non-linearity
        x = self.dropout(x)
        x = self.fc2(x)       # 3072 → 768
        x = self.dropout(x)
        return x

# Theoretical: "Each of the 12 identical transformer layers contains:
# Pre-LN → Multi-Head Self-Attention → residual connection
# Pre-LN → Feed-Forward MLP → residual connection"

In [ ]:
class TransformerLayer(nn.Module):
    def __init__(self, embed_dim=768, num_heads=12, mlp_ratio=4,
                 dropout=0.0, attn_dropout=0.0):
        super().__init__()
        # Pre-LayerNorm (applied BEFORE each sub-block)
        self.norm1 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.norm2 = nn.LayerNorm(embed_dim, eps=1e-6)

        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, attn_dropout)
        self.mlp  = FeedForwardMLP(embed_dim, mlp_ratio, dropout)

    def forward(self, x):
        # Pre-LN → Attention → residual
        x = x + self.attn(self.norm1(x))
        # Pre-LN → MLP → residual
        x = x + self.mlp(self.norm2(x))
        return x


# Theoretical pipeline:
# Patch Extraction → Patch Embedding → CLS Token →
# Positional Embedding → 12x Transformer Layers →
# Post LayerNorm → CLS token output (768-dim image embedding)

In [ ]:
class SigLIPVisionEncoder(nn.Module):
    def __init__(self,
                 image_size=224,
                 patch_size=16,
                 embed_dim=768,
                 num_layers=12,
                 num_heads=12,
                 mlp_ratio=4,
                 dropout=0.0):
        super().__init__()

        # Step 1 — Patch Embedding
        self.patch_embed = PatchEmbedding(image_size, patch_size, 3, embed_dim)
        num_patches = self.patch_embed.num_patches   # 196

        # Step 2 — CLS Token
        # Learnable vector prepended to patch sequence
        # Accumulates global image representation through attention
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Step 3 — Positional Embeddings
        # One vector per position: 1 CLS + 196 patches = 197
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(dropout)

        # Step 4 — 12 Transformer Layers
        self.layers = nn.ModuleList([
            TransformerLayer(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_layers)
        ])

        # Step 5 — Post LayerNorm (applied after all 12 layers)
        self.post_layernorm = nn.LayerNorm(embed_dim, eps=1e-6)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]

        # Step 1: Extract and embed patches
        x = self.patch_embed(x)   # (B, 196, 768)

        # Step 2: Prepend CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)   # (B, 1, 768)
        x = torch.cat([cls_tokens, x], dim=1)            # (B, 197, 768)

        # Step 3: Add positional embeddings
        x = x + self.pos_embed    # (B, 197, 768)
        x = self.pos_drop(x)

        # Step 4: Pass through all 12 transformer layers
        for layer in self.layers:
            x = layer(x)          # (B, 197, 768) at each step

        # Step 5: Post LayerNorm
        x = self.post_layernorm(x)   # (B, 197, 768)

        # Extract CLS token — the global image representation
        cls_output = x[:, 0]         # (B, 768)
        return cls_output



# Theoretical: "The 768-dim CLS output passes through:
# LayerNorm → Linear(768→512) → GELU → Dropout → Linear(512→17)"
# This is the only part that is trained from scratch.
# The vision encoder weights are loaded from SigLIP pretraining.

In [ ]:
class SigLIPClassifier(nn.Module):
    def __init__(self, num_classes=17):
        super().__init__()

        # Vision encoder — defined from scratch above
        self.vision_encoder = SigLIPVisionEncoder(
            image_size=224,
            patch_size=16,
            embed_dim=768,
            num_layers=12,
            num_heads=12,
            mlp_ratio=4,
        )

        # Classification head attached to CLS token output
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),           # stabilize CLS features
            nn.Linear(768, 512),         # bottleneck compression
            nn.GELU(),                   # non-linearity
            nn.Dropout(0.4),             # regularization
            nn.Linear(512, num_classes)  # 17 class logits
        )

    def forward(self, pixel_values):
        # Run image through the full pipeline
        cls_output = self.vision_encoder(pixel_values)   # (B, 768)
        return self.classifier(cls_output)               # (B, 17)

    def load_siglip_pretrained_weights(self, device='cpu'):
        """
        Loads pretrained SigLIP weights from HuggingFace into our
        hand-written architecture. Only the vision encoder weights
        are loaded — the classification head is always trained from scratch.
        """
        print("Loading SigLIP pretrained weights from HuggingFace...")
        pretrained = AutoModel.from_pretrained("google/siglip-base-patch16-224")
        pretrained_dict = pretrained.vision_model.state_dict()
        our_dict       = self.vision_encoder.state_dict()

        # Map HuggingFace weight names to our architecture names
        name_map = {}
        for k in pretrained_dict.keys():
            new_k = k
            # Patch embedding
            new_k = new_k.replace("embeddings.patch_embedding.weight",  "patch_embed.projection.weight")
            new_k = new_k.replace("embeddings.patch_embedding.bias",    "patch_embed.projection.bias")
            # CLS token and positional embedding
            new_k = new_k.replace("embeddings.class_embedding",         "cls_token")
            new_k = new_k.replace("embeddings.position_embedding.weight","pos_embed")
            # Transformer layers
            new_k = new_k.replace("encoder.layers.",                    "layers.")
            new_k = new_k.replace(".self_attn.q_proj.",                 ".attn.qkv.")   # merged below
            new_k = new_k.replace(".layer_norm1.",                      ".norm1.")
            new_k = new_k.replace(".layer_norm2.",                      ".norm2.")
            new_k = new_k.replace(".mlp.fc1.",                          ".mlp.fc1.")
            new_k = new_k.replace(".mlp.fc2.",                          ".mlp.fc2.")
            new_k = new_k.replace(".self_attn.out_proj.",               ".attn.proj.")
            # Post layernorm
            new_k = new_k.replace("post_layernorm.",                    "post_layernorm.")
            name_map[k] = new_k

        # Load matching weights
        matched, skipped = 0, 0
        for old_k, new_k in name_map.items():
            if new_k in our_dict and pretrained_dict[old_k].shape == our_dict[new_k].shape:
                our_dict[new_k] = pretrained_dict[old_k]
                matched += 1
            else:
                skipped += 1

        self.vision_encoder.load_state_dict(our_dict, strict=False)
        print(f"Loaded {matched} layers | Skipped {skipped} layers")
        print("Classification head randomly initialized — will be trained from scratch.")
        del pretrained

freezing

In [ ]:
def set_backbone_trainable(model, num_layers_to_unfreeze):
    # Freeze entire vision encoder
    for param in model.vision_encoder.parameters():
        param.requires_grad = False

    if num_layers_to_unfreeze > 0:
        # Unfreeze last N transformer layers
        for layer in model.vision_encoder.layers[-num_layers_to_unfreeze:]:
            for param in layer.parameters():
                param.requires_grad = True
        # Always unfreeze post layernorm
        for param in model.vision_encoder.post_layernorm.parameters():
            param.requires_grad = True

    # Classifier head always trainable
    for param in model.classifier.parameters():
        param.requires_grad = True


def get_optimizer(model, backbone_lr, head_lr, weight_decay):
    backbone_params = [p for n, p in model.named_parameters()
                       if "vision_encoder" in n and p.requires_grad]
    head_params     = [p for n, p in model.named_parameters()
                       if "classifier" in n and p.requires_grad]
    return torch.optim.AdamW([
        {"params": head_params,     "lr": head_lr},
        {"params": backbone_params, "lr": backbone_lr, "weight_decay": weight_decay}
    ], weight_decay=weight_decay)


In [ ]:
def main():
    data_root = r"C:\Migration files\black hole\new bylaw\spring 26\ai\project\cse-281-spring-26-scene-style-classification\StyleClassificationIndoors\StyleClassificationIndoors\train"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    SIGLIP_MEAN, SIGLIP_STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

    train_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=SIGLIP_MEAN, std=SIGLIP_STD),
        transforms.RandomErasing(p=0.2),
    ])

    val_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=SIGLIP_MEAN, std=SIGLIP_STD),
    ])

    full_dataset = datasets.ImageFolder(root=data_root)
    num_classes  = len(full_dataset.classes)
    indices      = list(range(len(full_dataset)))
    random.seed(42)
    random.shuffle(indices)
    split = int(0.85 * len(full_dataset))

    class ApplyTransform(torch.utils.data.Dataset):
        def __init__(self, dataset, transform=None):
            self.dataset   = dataset
            self.transform = transform
        def __getitem__(self, index):
            x, y = self.dataset[index]
            if self.transform: x = self.transform(x)
            return x, y
        def __len__(self): return len(self.dataset)

    train_loader = DataLoader(
        ApplyTransform(Subset(full_dataset, indices[:split]), train_transform),
        batch_size=32, shuffle=True, num_workers=4
    )
    val_loader = DataLoader(
        ApplyTransform(Subset(full_dataset, indices[split:]), val_transform),
        batch_size=32, shuffle=False, num_workers=4
    )

    # Build model and load SigLIP pretrained weights
    model = SigLIPClassifier(num_classes=num_classes).to(device)
    model.load_siglip_pretrained_weights(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    UNFREEZE_SCHEDULE = {1: 0, 4: 4, 8: 8}
    TOTAL_EPOCHS = 15
    best_val_acc = 0.0
    current_layers = -1

    print(f"Classes: {num_classes} | Device: {device}")

    for epoch in range(1, TOTAL_EPOCHS + 1):
        new_layers = next(v for k, v in sorted(UNFREEZE_SCHEDULE.items(), reverse=True) if epoch >= k)

        if new_layers != current_layers:
            current_layers = new_layers
            set_backbone_trainable(model, current_layers)
            optimizer = get_optimizer(model, backbone_lr=4e-6, head_lr=8e-5, weight_decay=0.05)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS)
            print(f"\n[Epoch {epoch}] Unfreezing {current_layers} layers. Backbone LR: 4e-6")

        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            output = model(imgs)
            loss   = criterion(output, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            correct    += (output.argmax(1) == lbls).sum().item()
            total      += lbls.size(0)

        scheduler.step()

        model.eval()
        val_preds, val_lbls = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs = imgs.to(device)
                val_preds.extend(model(imgs).argmax(1).cpu().numpy())
                val_lbls.extend(lbls.numpy())

        val_acc = balanced_accuracy_score(val_lbls, val_preds) * 100
        status  = "✓" if val_acc > best_val_acc else ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")

        print(f"Ep {epoch:02d} | Train Acc: {100*correct/total:.1f}% | Val Acc: {val_acc:.1f}% {status}")


if __name__ == "__main__":
    main()